# 🎓 AI Research Foundations - Module 20: Capstone GRPO Loss Walkthrough

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amirlankalm/ai-research-foundations/blob/main/20_Capstone_GRPO_Loss_Walkthrough/module_20.ipynb)

In this capstone walkthrough notebook, we implement and execute the core loss and group advantage normalization of **Group Relative Policy Optimization (GRPO)** from DeepSeekMath (arXiv:2402.03300).

In [ ]:
import re
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## 1. Rule-Based Mathematical Verifier
Deterministic verifier parsing `<think>...</think><answer>...</answer>` tags.

In [ ]:
def parse_and_verify(generation: str, ground_truth: int) -> tuple[float, bool]:
    reward = 0.0
    # Format bonus
    has_format = bool(re.search(r"<think>.*?</think>\s*<answer>.*?</answer>", generation, re.DOTALL))
    if has_format:
        reward += 0.2
    match = re.search(r"<answer>\s*([+-]?\d+)\s*</answer>", generation)
    if match and int(match.group(1)) == ground_truth:
        reward += 1.0
        return reward, True
    return reward, False

sample = "<think>2+2=4</think><answer>4</answer>"
r, ok = parse_and_verify(sample, 4)
print(f"Verifier test: Reward={r}, Correct={ok}")

## 2. Group Advantage Normalization (Critic-Free)
GRPO computes advantages purely from the empirical mean and standard deviation of responses sampled for each prompt.

In [ ]:
def compute_group_advantages(rewards: torch.Tensor, eps: float = 1e-4) -> torch.Tensor:
    mean = rewards.mean(dim=-1, keepdim=True)
    std = rewards.std(dim=-1, keepdim=True)
    return (rewards - mean) / (std + eps)

sample_rewards = torch.tensor([[1.2, 0.2, 0.0, 1.2]])
adv = compute_group_advantages(sample_rewards)
print("Sample Rewards:    ", sample_rewards[0].tolist())
print("Group Advantages:  ", [round(a, 3) for a in adv[0].tolist()])

## 3. GRPO Clipped Objective with KL Divergence

In [ ]:
def grpo_loss(pi_logps, old_logps, ref_logps, advantages, epsilon=0.2, beta=0.04):
    ratio = torch.exp(pi_logps - old_logps)
    surr1 = ratio * advantages
    surr2 = torch.clamp(ratio, 1.0 - epsilon, 1.0 + epsilon) * advantages
    policy_loss = -torch.min(surr1, surr2)
    kl = torch.exp(ref_logps - pi_logps) - (ref_logps - pi_logps) - 1.0
    total_loss = (policy_loss + beta * kl).mean()
    return total_loss, policy_loss.mean(), kl.mean()

print("GRPO loss function defined.")

## 4. End-to-End Walkthrough Optimization on Math Reasoning

In [ ]:
logits = nn.Parameter(torch.tensor([0.0, 0.0, 0.0, 0.0]))
ref_logits = torch.tensor([0.0, 0.0, 0.0, 0.0])
optimizer = torch.optim.Adam([logits], lr=0.1)
rewards = torch.tensor([1.2, 0.2, 0.0, 0.0])
adv = compute_group_advantages(rewards.unsqueeze(0)).squeeze(0)

print("=== Starting GRPO Walkthrough Optimization ===")
for step in range(1, 21):
    pi_logps = F.log_softmax(logits, dim=-1)
    old_logps = pi_logps.detach()
    ref_logps = F.log_softmax(ref_logits, dim=-1)
    
    loss, pol, kl = grpo_loss(pi_logps.unsqueeze(0), old_logps.unsqueeze(0), ref_logps.unsqueeze(0), adv.unsqueeze(0))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if step % 5 == 0 or step == 1:
        probs = F.softmax(logits, dim=-1)
        print(f"Step {step:02d} | Loss: {loss.item():.4f} | Correct Cand Prob: {probs[0].item():.4f}")

print("Capstone GRPO walkthrough complete!")